# Dataset

In [6]:
import json
import sys
import warnings
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")

## 1. Project paths

This cell finds the repository root, defines the dataset folder, and creates the output folders used by the notebook.

In [7]:
def rootpath():
    here = Path.cwd().resolve()

    for p in [here, *here.parents]:
        if (p / "document_generator").exists() and (p / "thesis_project").exists():
            return p

root = rootpath()

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from document_generator import generate_dataset
from thesis_project.dataset import DatasetManager
from thesis_project.shared import canon, direct, quasi

work = root / "notebooks" / "workspace"
results = root / "notebooks" / "results"
charts = root / "notebooks" / "charts"

dataName = "dataset"
dataRoot = work / dataName

for p in [work, results, charts]:
    p.mkdir(parents=True, exist_ok=True)

print(f"Repository: {root}")
print(f"Dataset:    {dataRoot}")

Repository: thesis-1
Dataset:    notebooks\workspace\dataset


## 2. Dataset settings

The dataset contains 100 synthetic documents based on three student-document templates. Set `regenerate = True` only when the dataset should be rebuilt.

In [8]:
regenerate = False

directLabels = [
    "FULL_NAME",
    "FIRST_NAME",
    "LAST_NAME",
    "Student number",
    "Institutional email",
]

quasiLabels = [
    "Course code",
    "Group label",
    "Programme",
    "Date",
    "Role",
]

labels = directLabels + quasiLabels

regions = [
    "cover",
    "header",
    "footer",
    "body",
    "body_table",
    "ocr",
]

cfg = {
    "dataset_name": dataName,
    "seed": 2026,
    "output_dir": str(work),

    "templates": [
        {
            "template_id": "agenda",
            "path": str(root / "document_generator" / "data" / "templates" / "agenda_template.docx"),
            "spec": str(root / "document_generator" / "data" / "templates" / "agenda_template.spec.json"),
            "weight": 1.0,
        },
        {
            "template_id": "minutes",
            "path": str(root / "document_generator" / "data" / "templates" / "minutes_template.docx"),
            "spec": str(root / "document_generator" / "data" / "templates" / "minutes_template.spec.json"),
            "weight": 1.0,
        },
        {
            "template_id": "project_plan",
            "path": str(root / "document_generator" / "data" / "templates" / "project_plan_template.docx"),
            "spec": str(root / "document_generator" / "data" / "templates" / "project_plan_template.spec.json"),
            "weight": 1.0,
        },
    ],

    "population": {
        "num_students": 3500,
        "num_teaching_assistants": 30,
        "num_coordinators": 3,
        "num_courses": 8,
        "groups_per_course": 6,
    },

    "documents": {
        "num_documents": 500,
        "pdf_ratio": 0.0,
        "pdf_converter_binary": "soffice",
    },

    "splits": {
        "train": 0.70,
        "val": 0.10,
        "test": 0.20,
    },

    "controller": {
        "typo_prob_student_number": 0.05,
        "typo_prob_email": 0.05,
        "header_prob": 0.5,
        "header_pii_prob": 0.20,
        "footer_prob": 0.5,
        "footer_pii_prob": 0.20,
    },
}

cfgPath = work / "generation-config.json"
cfgPath.write_text(json.dumps(cfg, indent=2), encoding="utf-8")

1512

## 3. Create or load the dataset

If the dataset folder already exists, the notebook reuses it. This keeps the later experiments stable.

In [9]:
if regenerate or not dataRoot.exists():
    generate_dataset(cfgPath)

dm = DatasetManager(dataRoot)
records = dm.load_manifest()

if not records:
    generate_dataset(cfgPath)
    records = dm.load_manifest()

print(f"Loaded documents: {len(records)}")

Loaded documents: 100


## 4. Build the documentation tables

The manifest table has one row per document.

The identifier coverage table has one row per annotated identifier mention.

In [10]:
def idclass(label):
    if direct(label):
        return "direct"
    if quasi(label):
        return "quasi"
    return "other"


def makemanifest(records):
    rows = []

    for r in records:
        d = dm.attach(r)

        rows.append({
            "docid": r.doc_id,
            "template": r.template_id,
            "split": r.split,
            "type": r.metadata.get("document_type", r.document_path.suffix.lstrip(".")),
            "blocks": len(d.blocks),
            "images": len(d.images),
        })

    return pd.DataFrame(rows)


def makecoverage(records):
    rows = []

    for r in records:
        d = dm.attach(r)
        ann = dm.load_annotations(d)

        for key, source in [
            ("text_entities", "text"),
            ("ocr_entities", "ocr"),
        ]:
            for e in ann.get(key, []):
                label = canon(e.get("label", ""))

                rows.append({
                    "docid": r.doc_id,
                    "template": r.template_id,
                    "split": r.split,
                    "source": source,
                    "region": e.get("region", ""),
                    "label": label,
                    "class": idclass(label),
                })

    return pd.DataFrame(rows)


manifest = makemanifest(records)
coverage = makecoverage(records)

manifestPath = results / "00-manifest.csv"
coveragePath = results / "00-identifier-coverage.csv"

manifest.to_csv(manifestPath, index=False)
coverage.to_csv(coveragePath, index=False)

print("Saved tables:")
print(f"- {manifestPath.relative_to(root)}")
print(f"- {coveragePath.relative_to(root)}")

Saved tables:
- notebooks\results\00-manifest.csv
- notebooks\results\00-identifier-coverage.csv


## 5. Dataset summary

This table gives the main dataset numbers used in the thesis methodology section.

In [11]:
summary = pd.DataFrame([
    {
        "documents": len(manifest),
        "templates": manifest["template"].nunique(),
        "train": int((manifest["split"] == "train").sum()),
        "validation": int((manifest["split"] == "val").sum()),
        "test": int((manifest["split"] == "test").sum()),
        "identifier mentions": len(coverage),
        "direct mentions": int((coverage["class"] == "direct").sum()),
        "quasi mentions": int((coverage["class"] == "quasi").sum()),
        "ocr mentions": int((coverage["source"] == "ocr").sum()),
    }
])

display(summary)

splitTable = (
    manifest
    .groupby(["template", "split"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

display(splitTable)

,documents,templates,train,validation,test,identifier mentions,direct mentions,quasi mentions,ocr mentions
0,100,3,73,11,16,4616,4186,430,152


split,template,test,train,val
0,agenda,6,24,4
1,minutes,3,30,5
2,project_plan,7,19,2
